# PYNQ-Z1 LTC2208 Dual-Channel Test
Capture both LTC2208 channels at 130 MSPS into DDR, estimate each channel frequency, and plot four cycles.

In [ ]:
from pynq import Overlay, MMIO
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

BITFILE = 'base_add.bit'
LTC2208_BASE = 0x40002000
sys.path.insert(0, str(Path('.').resolve()))
from pynqz1_diansai2_ltc2208 import LTC2208Capture, LTC2208_SAMPLE_HZ, signed_codes

overlay = Overlay(BITFILE)
dma = overlay.axi_dma_0
ltc2208 = LTC2208Capture(dma, base_addr=LTC2208_BASE)
print('Overlay:', BITFILE)
print('LTC2208:', ltc2208.status())

In [ ]:
# Connect the two external signals under test to LTC2208 channel A and B.

In [ ]:
# LTC2208 real acquisition at the full 130 MSPS.
# channel_mask: 0b01=A only, 0b10=B only, 0b11=both.
SAMPLE_COUNT = 262_144  # 2.017 ms at 130 MSPS; supports four cycles from about 2 kHz upward.
raw, adc_a, adc_b, elapsed = ltc2208.capture(
    sample_count=SAMPLE_COUNT, channel_mask=0b11, decimation=1, timeout_s=3.0
)
print('DMA elapsed = %.6f s' % elapsed)
print(ltc2208.status())

In [ ]:
# Estimate A/B dominant frequencies independently, then display four periods
# of each channel.
FREQ_MIN_HZ = 2_000       # Raise this if low-frequency drift dominates.
FREQ_MAX_HZ = 60_000_000  # LTC2208 test range, below 65 MHz Nyquist.

def estimate_tone(samples):
    signal = signed_codes(samples).astype(np.float64)
    signal -= np.mean(signal)
    n = len(signal)
    spectrum = np.abs(np.fft.rfft(signal * np.hanning(n)))
    freq_axis = np.fft.rfftfreq(n, d=1.0 / LTC2208_SAMPLE_HZ)
    valid = (freq_axis >= FREQ_MIN_HZ) & (freq_axis <= FREQ_MAX_HZ)
    if not np.any(valid):
        raise RuntimeError('No FFT bins in the requested frequency range.')
    peak_bin = np.flatnonzero(valid)[np.argmax(spectrum[valid])]
    bin_offset = 0.0
    if 0 < peak_bin < len(spectrum) - 1:
        left, center, right = spectrum[peak_bin - 1:peak_bin + 2]
        denom = left - 2.0 * center + right
        if abs(denom) > 1e-12:
            bin_offset = 0.5 * (left - right) / denom
    frequency_hz = (peak_bin + bin_offset) * LTC2208_SAMPLE_HZ / n
    return signal, frequency_hz

signals = {}
for channel_name, samples in (('A', adc_a), ('B', adc_b)):
    signal, frequency_hz = estimate_tone(samples)
    samples_per_cycle = LTC2208_SAMPLE_HZ / frequency_hz
    four_cycle_points = int(round(4.0 * samples_per_cycle))
    points = min(len(signal), max(16, four_cycle_points))
    signals[channel_name] = (signal, frequency_hz, points, four_cycle_points)
    print('LTC2208 %s: %.6f MHz (%.3f Hz), display %d samples / %.3f cycles' % (
        channel_name, frequency_hz / 1e6, frequency_hz, points, points / samples_per_cycle))
    if points < four_cycle_points:
        print('  Warning: capture is shorter than four cycles; increase SAMPLE_COUNT.')
print('FFT resolution: %.3f Hz' % (LTC2208_SAMPLE_HZ / len(adc_a)))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
for axis, channel_name in zip(axes, ('A', 'B')):
    signal, frequency_hz, points, _ = signals[channel_name]
    t_us = np.arange(points) / LTC2208_SAMPLE_HZ * 1e6
    axis.plot(t_us, signal[:points], label='LTC2208 %s: %.6f MHz' % (channel_name, frequency_hz / 1e6))
    axis.set_xlabel('Time (us)')
    axis.set_ylabel('ADC signed code, DC removed')
    axis.grid(True)
    axis.legend()
plt.show()